# Phase 3: Feature Engineering

Creating new features from existing ones to improve predictive power for house pricing.

## Task 1: Import Libraries and Load Preprocessed Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)

print('Libraries imported successfully!')

In [ ]:
# Load preprocessed data from Phase 2
X = pd.read_csv('train_preprocessed.csv')
y = pd.read_csv('train_target.csv')['SalePrice']
df_original = pd.read_csv('train_cleaned.csv')

print(f'X (preprocessed) shape : {X.shape}')
print(f'y (target) shape       : {y.shape}')
print(f'df_original shape      : {df_original.shape}')

## Task 2: Understanding Feature Engineering

Feature engineering involves creating new features that better represent the underlying problem.  
For house prices we apply the following techniques:

| Technique | Example |
|---|---|
| **Aggregation** | Total SF = basement + 1st + 2nd floor |
| **Ratio** | Living area / Lot area |
| **Temporal** | House age = year sold - year built |
| **Boolean** | Has garage (0/1) |
| **Interaction** | OverallQual x GrLivArea |
| **Polynomial** | OverallQual squared |

## Task 3: Create Aggregation Features

In [ ]:
# Working copy from original cleaned data
df_fe = df_original.copy()

print(f'Starting number of features: {df_fe.shape[1]}')

### Step 3.1 - Total Square Footage

In [ ]:
sf_cols = ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']
if all(c in df_fe.columns for c in sf_cols):
    df_fe['TotalSF'] = df_fe[sf_cols].sum(axis=1)
    print('Created: TotalSF (basement + 1st floor + 2nd floor)')

### Step 3.2 - Bathroom Features

In [ ]:
bath_cols = ['FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath']
if all(c in df_fe.columns for c in bath_cols):
    df_fe['TotalBathrooms'] = (
        df_fe['FullBath']
        + 0.5 * df_fe['HalfBath']
        + df_fe['BsmtFullBath']
        + 0.5 * df_fe['BsmtHalfBath']
    )
    print('Created: TotalBathrooms')

### Step 3.3 - Porch and Outdoor Features

In [ ]:
porch_cols = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
if all(c in df_fe.columns for c in porch_cols):
    df_fe['TotalPorchSF'] = df_fe[porch_cols].sum(axis=1)
    print('Created: TotalPorchSF')

outdoor_cols = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
if all(c in df_fe.columns for c in outdoor_cols):
    df_fe['TotalOutdoorSF'] = df_fe[outdoor_cols].sum(axis=1)
    print('Created: TotalOutdoorSF')

### Step 3.4 - Custom Aggregation Features

In [ ]:
# Total finished basement area
bsmt_fin_cols = ['BsmtFinSF1', 'BsmtFinSF2']
if all(c in df_fe.columns for c in bsmt_fin_cols):
    df_fe['TotalBsmtFinSF'] = df_fe[bsmt_fin_cols].sum(axis=1)
    print('Created: TotalBsmtFinSF')

# Overall quality + condition score
if 'OverallQual' in df_fe.columns and 'OverallCond' in df_fe.columns:
    df_fe['QualCondScore'] = df_fe['OverallQual'] + df_fe['OverallCond']
    print('Created: QualCondScore')

## Task 4: Create Ratio and Proportion Features

In [ ]:
# Living area to lot area
if 'GrLivArea' in df_fe.columns and 'LotArea' in df_fe.columns:
    df_fe['LivingArea_to_LotArea'] = df_fe['GrLivArea'] / df_fe['LotArea'].replace(0, np.nan)
    print('Created: LivingArea_to_LotArea')

# Basement ratio (what fraction of total area is basement)
if 'TotalBsmtSF' in df_fe.columns and 'TotalSF' in df_fe.columns:
    df_fe['BasementRatio'] = df_fe['TotalBsmtSF'] / df_fe['TotalSF'].replace(0, np.nan)
    print('Created: BasementRatio')

# Garage area to living area
if 'GarageArea' in df_fe.columns and 'GrLivArea' in df_fe.columns:
    df_fe['GarageArea_to_LivArea'] = df_fe['GarageArea'] / df_fe['GrLivArea'].replace(0, np.nan)
    print('Created: GarageArea_to_LivArea')

# Bedroom to total rooms ratio
if 'BedroomAbvGr' in df_fe.columns and 'TotRmsAbvGrd' in df_fe.columns:
    df_fe['Bedroom_to_TotalRooms'] = df_fe['BedroomAbvGr'] / df_fe['TotRmsAbvGrd'].replace(0, np.nan)
    print('Created: Bedroom_to_TotalRooms')

## Task 5: Create Temporal Features

In [ ]:
# House age at time of sale
if 'YrSold' in df_fe.columns and 'YearBuilt' in df_fe.columns:
    df_fe['HouseAge'] = (df_fe['YrSold'] - df_fe['YearBuilt']).clip(lower=0)
    print('Created: HouseAge')

# Years since remodel
if 'YrSold' in df_fe.columns and 'YearRemodAdd' in df_fe.columns:
    df_fe['YearsSinceRemodel'] = (df_fe['YrSold'] - df_fe['YearRemodAdd']).clip(lower=0)
    print('Created: YearsSinceRemodel')

# Garage age - GarageYrBlt=0 qaraj olmadığını bildirir (Phase 1-dən)
if 'YrSold' in df_fe.columns and 'GarageYrBlt' in df_fe.columns:
    has_garage = df_fe['GarageYrBlt'] > 0
    df_fe['GarageAge'] = 0  # default: qaraj yoxdur
    df_fe.loc[has_garage, 'GarageAge'] = (
        df_fe.loc[has_garage, 'YrSold'] - df_fe.loc[has_garage, 'GarageYrBlt']
    ).clip(lower=0)
    print('Created: GarageAge')

# Is newly built (sold within 5 years of construction)
if 'HouseAge' in df_fe.columns:
    df_fe['IsNewHouse'] = (df_fe['HouseAge'] <= 5).astype(int)
    print('Created: IsNewHouse')


## Task 6: Create Boolean / Indicator Features

In [ ]:
if '2ndFlrSF' in df_fe.columns:
    df_fe['HasSecondFloor'] = (df_fe['2ndFlrSF'] > 0).astype(int)
    print('Created: HasSecondFloor')

if 'TotalBsmtSF' in df_fe.columns:
    df_fe['HasBasement'] = (df_fe['TotalBsmtSF'] > 0).astype(int)
    print('Created: HasBasement')

if 'GarageArea' in df_fe.columns:
    df_fe['HasGarage'] = (df_fe['GarageArea'] > 0).astype(int)
    print('Created: HasGarage')

if 'Fireplaces' in df_fe.columns:
    df_fe['HasFireplace'] = (df_fe['Fireplaces'] > 0).astype(int)
    print('Created: HasFireplace')

if 'PoolArea' in df_fe.columns:
    df_fe['HasPool'] = (df_fe['PoolArea'] > 0).astype(int)
    print('Created: HasPool')

if 'TotalPorchSF' in df_fe.columns:
    df_fe['HasPorch'] = (df_fe['TotalPorchSF'] > 0).astype(int)
    print('Created: HasPorch')

if 'WoodDeckSF' in df_fe.columns:
    df_fe['HasDeck'] = (df_fe['WoodDeckSF'] > 0).astype(int)
    print('Created: HasDeck')

if 'YearRemodAdd' in df_fe.columns and 'YearBuilt' in df_fe.columns:
    df_fe['IsRemodeled'] = (df_fe['YearRemodAdd'] != df_fe['YearBuilt']).astype(int)
    print('Created: IsRemodeled')

## Task 7: Create Interaction Features

In [ ]:
# Overall quality x living area (high quality large homes are especially valuable)
if 'OverallQual' in df_fe.columns and 'GrLivArea' in df_fe.columns:
    df_fe['Quality_x_Area'] = df_fe['OverallQual'] * df_fe['GrLivArea']
    print('Created: Quality_x_Area')

# Overall quality x condition
if 'OverallQual' in df_fe.columns and 'OverallCond' in df_fe.columns:
    df_fe['Qual_x_Cond'] = df_fe['OverallQual'] * df_fe['OverallCond']
    print('Created: Qual_x_Cond')

# Quality x total square footage
if 'OverallQual' in df_fe.columns and 'TotalSF' in df_fe.columns:
    df_fe['Quality_x_TotalSF'] = df_fe['OverallQual'] * df_fe['TotalSF']
    print('Created: Quality_x_TotalSF')

# Bathrooms x bedrooms
if 'TotalBathrooms' in df_fe.columns and 'BedroomAbvGr' in df_fe.columns:
    df_fe['Bathrooms_x_Bedrooms'] = df_fe['TotalBathrooms'] * df_fe['BedroomAbvGr']
    print('Created: Bathrooms_x_Bedrooms')

## Task 8: Create Polynomial Features (Optional)

In [ ]:
# Quality squared - captures non-linear premium at high quality levels
if 'OverallQual' in df_fe.columns:
    df_fe['OverallQual_Squared'] = df_fe['OverallQual'] ** 2
    print('Created: OverallQual_Squared')

# GrLivArea squared - captures non-linear size premium
if 'GrLivArea' in df_fe.columns:
    df_fe['GrLivArea_Squared'] = df_fe['GrLivArea'] ** 2
    print('Created: GrLivArea_Squared')

## Task 9: Feature Summary and Analysis

In [ ]:
# Identify new features (columns added beyond original)
new_features = [c for c in df_fe.columns if c not in df_original.columns]

print(f'Original feature count : {len(df_original.columns)}')
print(f'New features created   : {len(new_features)}')
print(f'Total features         : {len(df_fe.columns)}')
print()
print('New features:')
for i, feat in enumerate(new_features, 1):
    print(f'  {i:2d}. {feat}')

In [ ]:
# Statistics of new features
if new_features:
    print('Descriptive statistics for new features:')
    display(df_fe[new_features].describe())

## Task 10: Analyze Feature Correlations with Target

In [ ]:
# Add SalePrice for correlation analysis (original, not log-transformed)
df_fe['SalePrice'] = df_original['SalePrice']

# Calculate correlations with target
correlations = df_fe.select_dtypes(include=[np.number]).corr()['SalePrice'].sort_values(ascending=False)

print('Top 20 features most correlated with SalePrice:')
print(correlations.head(21).to_string())
print()
print('Bottom 10 features (weakest / negative correlation):')
print(correlations.tail(10).to_string())

In [ ]:
# Heatmap of top 15 features vs SalePrice
top15 = [c for c in correlations.head(16).index if c != 'SalePrice'][:15]

corr_matrix = df_fe[top15 + ['SalePrice']].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap - Top 15 Features vs SalePrice')
plt.tight_layout()
plt.show()

### Analysis: Which New Features are Most Valuable?

In [ ]:
if new_features:
    # Filter to only new features that exist in correlations index
    new_feat_in_corr = [f for f in new_features if f in correlations.index]
    new_feat_corrs = correlations[new_feat_in_corr].sort_values(ascending=False)

    print('New feature correlations with SalePrice:')
    print(new_feat_corrs.to_string())

    plt.figure(figsize=(10, 6))
    new_feat_corrs.sort_values().plot(kind='barh', color='teal')
    plt.xlabel('Correlation with SalePrice')
    plt.title('New Feature Correlations with SalePrice')
    plt.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
    plt.tight_layout()
    plt.show()

## Task 11: Feature Selection (Optional)

In [ ]:
# Identify low-correlation features
threshold = 0.05
low_corr = [c for c in correlations.index if abs(correlations[c]) < threshold and c != 'SalePrice']

print(f'Features with |correlation| < {threshold}: {len(low_corr)}')
print('First 10:', low_corr[:10])

# Decision: keeping these features for now.
# Tree-based models handle low-correlation features well,
# and they may still carry weak non-linear signals.

In [ ]:
# Check for multicollinearity among top 20 features
top20 = [c for c in correlations.head(21).index if c != 'SalePrice'][:20]

corr_top20 = df_fe[top20].corr()

high_corr_pairs = []
for i, col_a in enumerate(corr_top20.columns):
    for col_b in corr_top20.columns[i + 1:]:
        val = corr_top20.loc[col_a, col_b]
        if abs(val) > 0.8:
            high_corr_pairs.append((col_a, col_b, round(val, 3)))

print(f'Highly correlated feature pairs (|r| > 0.8): {len(high_corr_pairs)}')
for a, b, v in high_corr_pairs:
    print(f'  {a} <-> {b} : {v}')

## Task 12: Apply Feature Engineering to Test Set

In [ ]:
import os

# test_preprocessed.csv Phase 2-dən gəlir - birbaşa onu yüklə
if os.path.exists('test_preprocessed.csv'):
    test_df = pd.read_csv('test_preprocessed.csv')
    print(f'Loaded preprocessed test data: {test_df.shape}')
elif os.path.exists('test_cleaned.csv'):
    test_df = pd.read_csv('test_cleaned.csv')
    print(f'Loaded cleaned test data: {test_df.shape}')
else:
    test_df = pd.read_csv('test.csv')
    print(f'Loaded raw test data: {test_df.shape}')

# Missing value imputation
for col in test_df.select_dtypes(include=[np.number]).columns:
    test_df[col] = test_df[col].fillna(test_df[col].median())
for col in test_df.select_dtypes(include=['object', 'string', 'category']).columns:
    mode_vals = test_df[col].mode()
    if len(mode_vals) > 0:
        test_df[col] = test_df[col].fillna(mode_vals[0])

print(f'Missing values after imputation: {test_df.isnull().sum().sum()}')


In [ ]:
def apply_feature_engineering(df):
    """Apply all feature engineering transformations to a dataframe."""
    d = df.copy()

    # Aggregation
    if all(c in d.columns for c in ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']):
        d['TotalSF'] = d['TotalBsmtSF'] + d['1stFlrSF'] + d['2ndFlrSF']

    if all(c in d.columns for c in ['FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath']):
        d['TotalBathrooms'] = d['FullBath'] + 0.5*d['HalfBath'] + d['BsmtFullBath'] + 0.5*d['BsmtHalfBath']

    porch = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
    if all(c in d.columns for c in porch):
        d['TotalPorchSF'] = d[porch].sum(axis=1)

    outdoor = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
    if all(c in d.columns for c in outdoor):
        d['TotalOutdoorSF'] = d[outdoor].sum(axis=1)

    if all(c in d.columns for c in ['BsmtFinSF1', 'BsmtFinSF2']):
        d['TotalBsmtFinSF'] = d['BsmtFinSF1'] + d['BsmtFinSF2']

    if all(c in d.columns for c in ['OverallQual', 'OverallCond']):
        d['QualCondScore'] = d['OverallQual'] + d['OverallCond']

    # Ratios
    if all(c in d.columns for c in ['GrLivArea', 'LotArea']):
        d['LivingArea_to_LotArea'] = d['GrLivArea'] / d['LotArea'].replace(0, np.nan)

    if 'TotalBsmtSF' in d.columns and 'TotalSF' in d.columns:
        d['BasementRatio'] = d['TotalBsmtSF'] / d['TotalSF'].replace(0, np.nan)

    if all(c in d.columns for c in ['GarageArea', 'GrLivArea']):
        d['GarageArea_to_LivArea'] = d['GarageArea'] / d['GrLivArea'].replace(0, np.nan)

    if all(c in d.columns for c in ['BedroomAbvGr', 'TotRmsAbvGrd']):
        d['Bedroom_to_TotalRooms'] = d['BedroomAbvGr'] / d['TotRmsAbvGrd'].replace(0, np.nan)

    # Temporal
    if all(c in d.columns for c in ['YrSold', 'YearBuilt']):
        yr_sold = pd.to_numeric(d['YrSold'], errors='coerce')
        yr_built = pd.to_numeric(d['YearBuilt'], errors='coerce')
        d['HouseAge'] = (yr_sold - yr_built).clip(lower=0)

    if all(c in d.columns for c in ['YrSold', 'YearRemodAdd']):
        yr_sold = pd.to_numeric(d['YrSold'], errors='coerce')
        yr_remod = pd.to_numeric(d['YearRemodAdd'], errors='coerce')
        d['YearsSinceRemodel'] = (yr_sold - yr_remod).clip(lower=0)

    if all(c in d.columns for c in ['YrSold', 'GarageYrBlt']):
        yr_sold = pd.to_numeric(d['YrSold'], errors='coerce')
        garage_yr = pd.to_numeric(d['GarageYrBlt'], errors='coerce').fillna(0)
        has_garage = garage_yr > 0
        d['GarageAge'] = 0.0
        d.loc[has_garage, 'GarageAge'] = (
            yr_sold[has_garage] - garage_yr[has_garage]
        ).clip(lower=0)

    if 'HouseAge' in d.columns:
        d['IsNewHouse'] = (pd.to_numeric(d['HouseAge'], errors='coerce').fillna(0) <= 5).astype(int)

    # Boolean
    bool_map = {
        'HasSecondFloor': '2ndFlrSF',
        'HasBasement':    'TotalBsmtSF',
        'HasGarage':      'GarageArea',
        'HasFireplace':   'Fireplaces',
        'HasPool':        'PoolArea',
        'HasDeck':        'WoodDeckSF',
    }
    for feat, col in bool_map.items():
        if col in d.columns:
            # pd.to_numeric: scaled/float sütunlarda da işləyir
            d[feat] = (pd.to_numeric(d[col], errors='coerce').fillna(0) > 0).astype(int)

    if 'TotalPorchSF' in d.columns:
        d['HasPorch'] = (pd.to_numeric(d['TotalPorchSF'], errors='coerce').fillna(0) > 0).astype(int)

    if all(c in d.columns for c in ['YearRemodAdd', 'YearBuilt']):
        d['IsRemodeled'] = (d['YearRemodAdd'] != d['YearBuilt']).astype(int)

    # Interactions
    if all(c in d.columns for c in ['OverallQual', 'GrLivArea']):
        d['Quality_x_Area'] = d['OverallQual'] * d['GrLivArea']

    if all(c in d.columns for c in ['OverallQual', 'OverallCond']):
        d['Qual_x_Cond'] = d['OverallQual'] * d['OverallCond']

    if all(c in d.columns for c in ['OverallQual', 'TotalSF']):
        d['Quality_x_TotalSF'] = d['OverallQual'] * d['TotalSF']

    if all(c in d.columns for c in ['TotalBathrooms', 'BedroomAbvGr']):
        d['Bathrooms_x_Bedrooms'] = d['TotalBathrooms'] * d['BedroomAbvGr']

    # Polynomial
    if 'OverallQual' in d.columns:
        d['OverallQual_Squared'] = d['OverallQual'] ** 2

    if 'GrLivArea' in d.columns:
        d['GrLivArea_Squared'] = d['GrLivArea'] ** 2

    return d


test_fe = apply_feature_engineering(test_df)
print(f'Test set after feature engineering: {test_fe.shape}')
print('New test features:', [c for c in test_fe.columns if c not in test_df.columns])

In [ ]:
test_fe.to_csv('test_feature_engineered.csv', index=False)
print('Saved: test_feature_engineered.csv')

## Task 13: Final Feature Engineering Summary

In [ ]:
# Final dataset includes SalePrice
df_fe_final = df_fe.copy()

print('=' * 50)
print('FEATURE ENGINEERING SUMMARY')
print('=' * 50)
print(f'Original feature count : {len(df_original.columns) - 1}  (excl. SalePrice)')
print(f'New features created   : {len(new_features)}')
print(f'Total features         : {len(df_fe_final.columns)}')
print(f'Total samples          : {len(df_fe_final)}')
print(f'Missing values         : {df_fe_final.isnull().sum().sum()}')

In [ ]:
# Save feature-engineered training dataset
df_fe_final.to_csv('train_feature_engineered.csv', index=False)
print('Saved: train_feature_engineered.csv')

# Save list of new features for documentation
with open('new_features_list.txt', 'w') as f:
    f.write('New Features Created:\n')
    f.write('-' * 40 + '\n')
    for i, feat in enumerate(new_features, 1):
        f.write(f'{i:2d}. {feat}\n')
print('Saved: new_features_list.txt')

## Summary and Reflection

**What we accomplished:**
- **Aggregation**: `TotalSF`, `TotalBathrooms`, `TotalPorchSF`, `TotalOutdoorSF`, `TotalBsmtFinSF`, `QualCondScore`
- **Ratios**: `LivingArea_to_LotArea`, `BasementRatio`, `GarageArea_to_LivArea`, `Bedroom_to_TotalRooms`
- **Temporal**: `HouseAge`, `YearsSinceRemodel`, `GarageAge`, `IsNewHouse`
- **Boolean**: `HasSecondFloor`, `HasBasement`, `HasGarage`, `HasFireplace`, `HasPool`, `HasPorch`, `HasDeck`, `IsRemodeled`
- **Interactions**: `Quality_x_Area`, `Qual_x_Cond`, `Quality_x_TotalSF`, `Bathrooms_x_Bedrooms`
- **Polynomial**: `OverallQual_Squared`, `GrLivArea_Squared`
- Applied identical transformations to the test set via `apply_feature_engineering()`

**Key takeaways:**
- Interaction features (`Quality_x_Area`, `Quality_x_TotalSF`) typically outperform individual components
- Temporal features like `HouseAge` capture depreciation invisible in raw year columns
- The `apply_feature_engineering()` function ensures train/test consistency and reproducibility

**Next step:** Phase 4 - Exploratory Data Analysis to visualise distributions and validate these engineering decisions.